# Análise de Desempenho: PostgreSQL (JSONB) vs MongoDB

Notebook centralizado do experimento: configura o ambiente, cria a estrutura dos bancos, gera e insere a massa de dados de teste, e roda as consultas comparativas — tudo em um único lugar, para facilitar a reprodutibilidade.

## 1. Configuração do Ambiente

Carrega as variáveis de conexão a partir dos arquivos `.env` de cada implementação e prepara os clientes de conexão (`SQLAlchemy` para o PostgreSQL, `PyMongo` para o MongoDB).

In [ ]:
import os
import json
import uuid
import random
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from pymongo import MongoClient
from bson import ObjectId
import pandas as pd

# Carrega as variáveis de ambiente dos dois projetos
load_dotenv("../catalogo-postgres/.env")
load_dotenv("../catalogo-mongo/.env")

# --- Conexão PostgreSQL (via SQLAlchemy) ---
pg_user = os.getenv("PGUSER")
pg_password = os.getenv("PGPASSWORD")
pg_db = os.getenv("PGDATABASE")
pg_host = os.getenv("PGHOST")
pg_port = os.getenv("PGPORT")

engine_pg = create_engine(f"postgresql://{pg_user}:{pg_password}@{pg_host}:{pg_port}/{pg_db}")

# --- Conexão MongoDB (via PyMongo) ---
mongo_uri = os.getenv("MONGO_URI")
mongo_db_name = os.getenv("MONGO_DB")

client_mongo = MongoClient(mongo_uri)
db_mongo = client_mongo[mongo_db_name]

random.seed(42)  # seed fixa, para reprodutibilidade da massa de dados

print("Conexões configuradas.")

## 2. Estrutura do Banco (Schema)

Cria as tabelas do PostgreSQL (`restaurantes`, `itens`) com as constraints definidas na modelagem (chave primária nomeada, chave estrangeira com `ON DELETE CASCADE`). No MongoDB, as coleções não exigem criação explícita — apenas são limpas aqui para permitir uma reexecução limpa do notebook.

**Atenção:** com `RESETAR_DADOS = True`, todos os dados existentes nas duas bases são apagados antes de recriar a estrutura.

In [ ]:
RESETAR_DADOS = True  # mude para False para preservar dados já existentes

DDL_POSTGRES = """
DROP TABLE IF EXISTS itens CASCADE;
DROP TABLE IF EXISTS restaurantes CASCADE;

CREATE TABLE restaurantes (
  id          UUID DEFAULT gen_random_uuid(),
  nome        VARCHAR(255) NOT NULL,
  categoria   VARCHAR(100),
  ativo       BOOLEAN NOT NULL DEFAULT true,
  criado_em   TIMESTAMP NOT NULL DEFAULT now(),

  CONSTRAINT pk_restaurante PRIMARY KEY (id)
);

CREATE TABLE itens (
  id                    UUID DEFAULT gen_random_uuid(),
  restaurante_id        UUID NOT NULL,
  nome                  VARCHAR(255) NOT NULL,
  categoria             VARCHAR(100) NOT NULL,
  disponivel            BOOLEAN NOT NULL DEFAULT true,
  preco                 NUMERIC(10,2) NOT NULL,
  atributos_variaveis   JSONB NOT NULL DEFAULT '{}'::jsonb,
  criado_em             TIMESTAMP NOT NULL DEFAULT now(),
  atualizado_em         TIMESTAMP NOT NULL DEFAULT now(),

  CONSTRAINT pk_item PRIMARY KEY (id),
  CONSTRAINT fk_item_restaurante FOREIGN KEY (restaurante_id)
    REFERENCES restaurantes(id) ON DELETE CASCADE
);
"""

if RESETAR_DADOS:
    with engine_pg.begin() as conn:
        conn.execute(text(DDL_POSTGRES))
    print("✅ Tabelas do PostgreSQL recriadas.")

    db_mongo.restaurantes.drop()
    db_mongo.itens.drop()
    print("✅ Coleções do MongoDB limpas (serão recriadas na primeira inserção).")
else:
    print("RESETAR_DADOS = False — estrutura/dados existentes preservados.")

## 3. Geração da Massa de Dados

Gera restaurantes e itens de forma sintética, com os atributos variáveis mudando conforme a categoria do item (`pizza`, `bebida`, `sobremesa`, `lanche`) — reproduzindo a heterogeneidade real de um catálogo, conforme a modelagem de dados do TCC. Os mesmos dados (nomes, categorias) são usados nos dois bancos; apenas o formato do identificador muda (`UUID` no PostgreSQL, `ObjectId` no MongoDB).

In [ ]:
NUM_RESTAURANTES = 5
NUM_ITENS = 100

CATEGORIAS_REST = ['italiana', 'brasileira', 'árabe', 'mexicana']
CATEGORIAS_ITEM = ['pizza', 'bebida', 'sobremesa', 'lanche']

NOMES_RESTAURANTES = ['Cantina Nonna', 'Sabor Caseiro', 'Oásis Árabe', 'Casa do Taco', 'Empório Brasil']
NOMES_POR_CATEGORIA = {
    'pizza': ['Margherita', 'Calabresa', 'Quatro Queijos', 'Frango com Catupiry', 'Portuguesa'],
    'bebida': ['Coca-Cola Lata', 'Guaraná Antarctica', 'Suco de Laranja', 'Água Mineral'],
    'sobremesa': ['Petit Gateau', 'Pudim', 'Mousse de Maracujá', 'Brownie'],
    'lanche': ['X-Burger', 'X-Salada', 'X-Bacon', 'X-Tudo'],
}
TAGS = ['vegetariano', 'mais vendido', 'sem glúten', 'picante']
INGREDIENTES = ['queijo', 'bacon', 'alface', 'tomate', 'ovo', 'molho especial']

def sortear_tags():
    return [t for t in TAGS if random.random() > 0.7]

def sortear_ingredientes():
    escolhidos = [i for i in INGREDIENTES if random.random() > 0.5]
    return escolhidos if len(escolhidos) >= 2 else INGREDIENTES[:2]

def gerar_promocao():
    return {"descontoPercentual": random.randint(5, 30), "validoAte": "2026-12-31"}

def gerar_atributos(categoria):
    if categoria == 'pizza':
        attrs = {"opcoes": [{"nome": "Borda recheada", "precoAdicional": 8.0}],
                 "tamanhos": ["média", "grande", "família"], "tags": sortear_tags()}
        if random.random() > 0.6:
            attrs["promocao"] = gerar_promocao()
        return attrs
    if categoria == 'bebida':
        return {"volumeMl": random.choice([350, 500, 600]),
                "gelada": random.random() > 0.2, "tags": sortear_tags()}
    if categoria == 'sobremesa':
        attrs = {"informacaoNutricional": {"calorias": random.randint(200, 500),
                                             "contemGluten": random.random() > 0.5}}
        if random.random() > 0.7:
            attrs["promocao"] = gerar_promocao()
        return attrs
    # lanche
    attrs = {"ingredientes": sortear_ingredientes(),
             "tamanho": random.choice(["único", "médio", "grande"]),
             "opcoes": [{"nome": "Sem cebola", "precoAdicional": 0}],
             "tags": sortear_tags()}
    if random.random() > 0.65:
        attrs["promocao"] = gerar_promocao()
    return attrs

def gerar_preco(categoria):
    faixas = {'pizza': (35, 70), 'bebida': (5, 15), 'sobremesa': (12, 25), 'lanche': (18, 35)}
    lo, hi = faixas[categoria]
    return round(random.uniform(lo, hi), 2)

# --- Restaurantes: mesmo nome/categoria, ID em dois formatos (pg / mongo) ---
restaurantes = []
for i in range(NUM_RESTAURANTES):
    restaurantes.append({
        "pg_id": str(uuid.uuid4()),
        "mongo_id": ObjectId(),
        "nome": NOMES_RESTAURANTES[i % len(NOMES_RESTAURANTES)],
        "categoria": CATEGORIAS_REST[i % len(CATEGORIAS_REST)],
        "ativo": True,
    })

# --- Itens: distribuídos ciclicamente entre os restaurantes ---
itens = []
for i in range(NUM_ITENS):
    restaurante = restaurantes[i % NUM_RESTAURANTES]
    categoria = CATEGORIAS_ITEM[i % len(CATEGORIAS_ITEM)]
    nome_base = random.choice(NOMES_POR_CATEGORIA[categoria])
    itens.append({
        "restaurante": restaurante,
        "nome": f"{nome_base} #{i+1}",
        "categoria": categoria,
        "disponivel": random.random() > 0.1,
        "preco": gerar_preco(categoria),
        "atributos": gerar_atributos(categoria),
    })

print(f"Gerados: {len(restaurantes)} restaurantes e {len(itens)} itens.")

## 4. Inserção dos Dados

Insere a massa de dados gerada nos dois bancos.

In [ ]:
# --- Inserção no PostgreSQL ---
with engine_pg.begin() as conn:
    for r in restaurantes:
        conn.execute(text("""
            INSERT INTO restaurantes (id, nome, categoria, ativo)
            VALUES (:id, :nome, :categoria, :ativo)
        """), {"id": r["pg_id"], "nome": r["nome"], "categoria": r["categoria"], "ativo": r["ativo"]})

    for it in itens:
        conn.execute(text("""
            INSERT INTO itens (restaurante_id, nome, categoria, disponivel, preco, atributos_variaveis)
            VALUES (:restaurante_id, :nome, :categoria, :disponivel, :preco, CAST(:atributos AS JSONB))
        """), {
            "restaurante_id": it["restaurante"]["pg_id"],
            "nome": it["nome"],
            "categoria": it["categoria"],
            "disponivel": it["disponivel"],
            "preco": it["preco"],
            "atributos": json.dumps(it["atributos"], ensure_ascii=False),
        })

print(f"✅ PostgreSQL: {len(restaurantes)} restaurantes e {len(itens)} itens inseridos.")

In [ ]:
# --- Inserção no MongoDB ---
docs_restaurantes = [
    {"_id": r["mongo_id"], "nome": r["nome"], "categoria": r["categoria"], "ativo": r["ativo"]}
    for r in restaurantes
]
db_mongo.restaurantes.insert_many(docs_restaurantes)

docs_itens = []
for it in itens:
    doc = {
        "restauranteId": it["restaurante"]["mongo_id"],
        "nome": it["nome"],
        "categoria": it["categoria"],
        "disponivel": it["disponivel"],
        "preco": it["preco"],
    }
    doc.update(it["atributos"])  # campos variáveis direto no documento
    docs_itens.append(doc)

db_mongo.itens.insert_many(docs_itens)

print(f"✅ MongoDB: {len(docs_restaurantes)} restaurantes e {len(docs_itens)} itens inseridos.")

## 5. Validação

Confirma que os dois bancos ficaram com a mesma quantidade de registros.

In [ ]:
with engine_pg.connect() as conn:
    contagem_pg = conn.execute(text("SELECT COUNT(*) FROM itens")).scalar()
print(f"PostgreSQL: {contagem_pg} itens")

contagem_mongo = db_mongo.itens.count_documents({})
print(f"MongoDB: {contagem_mongo} itens")

assert contagem_pg == contagem_mongo == NUM_ITENS, "Contagem divergente entre os bancos!"
print("✅ Os dois bancos estão consistentes.")

## 6. Primeira Consulta Comparativa

Busca os itens da categoria `pizza` nos dois bancos, como teste funcional antes de partirmos para a medição de tempo e para os cenários de carga.

In [ ]:
# PostgreSQL
df_pg = pd.read_sql(
    "SELECT nome, categoria, preco FROM itens WHERE categoria = 'pizza'",
    engine_pg
)
print(f"PostgreSQL: {len(df_pg)} pizzas encontradas")
df_pg.head()

In [ ]:
# MongoDB
cursor = db_mongo.itens.find(
    {"categoria": "pizza"},
    {"_id": 0, "nome": 1, "categoria": 1, "preco": 1}
)
df_mongo = pd.DataFrame(list(cursor))
print(f"MongoDB: {len(df_mongo)} pizzas encontradas")
df_mongo.head()